# Shape Mismatch Leak — bubbliiiing/classic-convolution-network

**Smell (`resnet50.py` line 125):** `Flatten()` is used directly before `Dense(1000)`. After the final `AveragePooling2D((7,7))`, the feature map is `(1,1,2048)`. Flattening it produces a 2048-d vector, then `Dense(classes)` creates a `2048×classes` weight matrix — wasteful and memory-heavy.

**Fix:** Replace `Flatten() → Dense(classes)` with `GlobalAveragePooling2D() → Dense(classes)`. GAP performs the spatial reduction correctly without a redundant reshape, using far less intermediate memory.

In [ ]:
!pip install -q codecarbon

In [ ]:
import codecarbon, os, json
_dir = os.path.join(os.path.dirname(codecarbon.__file__), 'data', 'private_infra')
os.makedirs(_dir, exist_ok=True)
_p = os.path.join(_dir, 'nordic_emissions.json')
if not os.path.exists(_p):
    with open(_p, 'w') as f: json.dump({}, f)
print('CodeCarbon ready')

In [ ]:
import gc, os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import (Input, Conv2D, BatchNormalization, Activation,
                                      Add, MaxPooling2D, AveragePooling2D,
                                      GlobalAveragePooling2D, Flatten, Dense,
                                      ZeroPadding2D)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from codecarbon import EmissionsTracker

os.makedirs('results', exist_ok=True)
N_RUNS = 10
EPOCHS = 5
BATCH  = 64

(x_tr, y_tr), (x_te, y_te) = cifar10.load_data()
x_tr = x_tr.astype('float32') / 255.0
x_te = x_te.astype('float32') / 255.0
y_tr_oh = to_categorical(y_tr, 10)
y_te_oh = to_categorical(y_te, 10)
print('CIFAR-10 loaded')

In [ ]:
# Shared residual block (from bubbliiiing resnet50.py)
def identity_block(x, filters, stage, block):
    f1, f2, f3 = filters
    x_skip = x
    x = Conv2D(f1, (1,1), name=f'res{stage}{block}_a')(x)
    x = BatchNormalization()(x); x = Activation('relu')(x)
    x = Conv2D(f2, (3,3), padding='same', name=f'res{stage}{block}_b')(x)
    x = BatchNormalization()(x); x = Activation('relu')(x)
    x = Conv2D(f3, (1,1), name=f'res{stage}{block}_c')(x)
    x = BatchNormalization()(x)
    x = Add()([x, x_skip]); x = Activation('relu')(x)
    return x

def conv_block(x, filters, stage, block, strides=(2,2)):
    f1, f2, f3 = filters
    x_skip = x
    x = Conv2D(f1, (1,1), strides=strides, name=f'res{stage}{block}_ca')(x)
    x = BatchNormalization()(x); x = Activation('relu')(x)
    x = Conv2D(f2, (3,3), padding='same', name=f'res{stage}{block}_cb')(x)
    x = BatchNormalization()(x); x = Activation('relu')(x)
    x = Conv2D(f3, (1,1), name=f'res{stage}{block}_cc')(x)
    x = BatchNormalization()(x)
    x_skip = Conv2D(f3,(1,1),strides=strides,name=f'res{stage}{block}_cs')(x_skip)
    x_skip = BatchNormalization()(x_skip)
    x = Add()([x, x_skip]); x = Activation('relu')(x)
    return x

def build_smelly_resnet50(input_shape=(32,32,3), classes=10):
    """bubbliiiing resnet50.py — SMELL: Flatten() before Dense()"""
    inp = Input(shape=input_shape)
    x = Conv2D(64, (3,3), padding='same', name='conv1')(inp)
    x = BatchNormalization()(x); x = Activation('relu')(x)
    x = conv_block(x, [64,64,256], 2, 'a', strides=(1,1))
    x = identity_block(x, [64,64,256], 2, 'b')
    x = conv_block(x, [128,128,512], 3, 'a')
    x = identity_block(x, [128,128,512], 3, 'b')
    x = AveragePooling2D((2,2), name='avg_pool')(x)
    # ❌ SMELL: Flatten then Dense — unnecessary reshape, large weight matrix
    x = Flatten()(x)                              # line 125 in resnet50.py
    x = Dense(classes, activation='softmax', name='fc')(x)
    model = Model(inp, x)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

def build_clean_resnet50(input_shape=(32,32,3), classes=10):
    """Fixed: GlobalAveragePooling2D replaces Flatten()"""
    inp = Input(shape=input_shape)
    x = Conv2D(64, (3,3), padding='same', name='conv1')(inp)
    x = BatchNormalization()(x); x = Activation('relu')(x)
    x = conv_block(x, [64,64,256], 2, 'a', strides=(1,1))
    x = identity_block(x, [64,64,256], 2, 'b')
    x = conv_block(x, [128,128,512], 3, 'a')
    x = identity_block(x, [128,128,512], 3, 'b')
    # ✅ FIX: GlobalAveragePooling2D — spatial reduction without intermediate tensor
    x = GlobalAveragePooling2D(name='gap')(x)
    x = Dense(classes, activation='softmax', name='fc')(x)
    model = Model(inp, x)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

print('Model builders ready')

## BEFORE — Smell Active (Flatten → Dense)

In [ ]:
results_before = []
es = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

for run in range(1, N_RUNS + 1):
    print(f'\n[BEFORE] Run {run}/{N_RUNS}')
    K.clear_session(); gc.collect()
    tracker = EmissionsTracker(
        project_name=f'bubbliiiing_resnet_before_run{run}',
        save_to_file=False, log_level='error'
    )
    tracker.start()
    model = build_smelly_resnet50()
    history = model.fit(x_tr, y_tr_oh, epochs=EPOCHS, batch_size=BATCH,
                        validation_data=(x_te, y_te_oh), callbacks=[es], verbose=1)
    _, acc = model.evaluate(x_te, y_te_oh, verbose=0)
    tracker.stop()
    em = tracker.final_emissions_data
    results_before.append({
        'run': run, 'accuracy': round(float(acc),4),
        'epochs_run': len(history.history['loss']),
        'energy_kWh': round(em.energy_consumed,8),
        'cpu_energy_kWh': round(em.cpu_energy,8),
        'gpu_energy_kWh': round(em.gpu_energy,8),
        'ram_energy_kWh': round(em.ram_energy,8),
        'emissions_kgCO2': round(em.emissions,8),
        'duration_s': round(em.duration,2),
    })
    print(f'  acc={acc:.4f}  energy={em.energy_consumed:.6f} kWh  CO2={em.emissions:.6f} kg  t={em.duration:.1f}s')
    del model; gc.collect()

df_before = pd.DataFrame(results_before)
df_before.to_csv('results/bubbliiiing_resnet50_before.csv', index=False)
print('\n--- BEFORE means ---')
print(df_before.mean(numeric_only=True))
print('Saved results/bubbliiiing_resnet50_before.csv')

## AFTER — Smell Fixed (GlobalAveragePooling2D → Dense)

In [ ]:
results_after = []

for run in range(1, N_RUNS + 1):
    print(f'\n[AFTER] Run {run}/{N_RUNS}')
    K.clear_session(); gc.collect()
    tracker = EmissionsTracker(
        project_name=f'bubbliiiing_resnet_after_run{run}',
        save_to_file=False, log_level='error'
    )
    tracker.start()
    model = build_clean_resnet50()
    history = model.fit(x_tr, y_tr_oh, epochs=EPOCHS, batch_size=BATCH,
                        validation_data=(x_te, y_te_oh), callbacks=[es], verbose=1)
    _, acc = model.evaluate(x_te, y_te_oh, verbose=0)
    tracker.stop()
    em = tracker.final_emissions_data
    results_after.append({
        'run': run, 'accuracy': round(float(acc),4),
        'epochs_run': len(history.history['loss']),
        'energy_kWh': round(em.energy_consumed,8),
        'cpu_energy_kWh': round(em.cpu_energy,8),
        'gpu_energy_kWh': round(em.gpu_energy,8),
        'ram_energy_kWh': round(em.ram_energy,8),
        'emissions_kgCO2': round(em.emissions,8),
        'duration_s': round(em.duration,2),
    })
    print(f'  acc={acc:.4f}  energy={em.energy_consumed:.6f} kWh  CO2={em.emissions:.6f} kg  t={em.duration:.1f}s')
    del model; gc.collect()

df_after = pd.DataFrame(results_after)
df_after.to_csv('results/bubbliiiing_resnet50_after.csv', index=False)
print('\n--- AFTER means ---')
print(df_after.mean(numeric_only=True))
print('Saved results/bubbliiiing_resnet50_after.csv')

In [ ]:
print('\n=== SUMMARY: bubbliiiing/classic-convolution-network ===')
print(f"BEFORE avg energy : {df_before['energy_kWh'].mean():.6f} kWh")
print(f"AFTER  avg energy : {df_after['energy_kWh'].mean():.6f} kWh")
print(f"BEFORE avg CO2    : {df_before['emissions_kgCO2'].mean():.6f} kg")
print(f"AFTER  avg CO2    : {df_after['emissions_kgCO2'].mean():.6f} kg")
print(f"BEFORE avg time   : {df_before['duration_s'].mean():.1f} s")
print(f"AFTER  avg time   : {df_after['duration_s'].mean():.1f} s")